In [8]:
from openai import OpenAI
import json
# Configured by environment variables
client = OpenAI(base_url="http://localhost:1234/v1", api_key="not-needed")

In [2]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_today_weather",
            "description": "Get a description of today's weather",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City Name"
                    },
                    "period": {
                        "type": "number",
                        "description": "1 for morning, 2 for afternoon"
                    }
                },
            },
            "returns": {
                "type": "string",
                "description": "Return string containing the weather description, and the temperature."
            }
        }
    },
]

content = [
    {
        "type": "text",
        "text": "What is the weather like this morning in San Francisco?",
    }
]

messages = [
    {
        "role": "user",
        "content": content,
    }
]

In [3]:
chat_response = client.chat.completions.create(
  model="Ternary-Bonsai-27B-Q2_0.gguf",
  messages=messages,
  tools=tools,
  max_tokens=4096,
  temperature=0.7,
  top_p=0.95,
  extra_body={
      "top_k": 20,
      # "enable_thinking": False,
  },
)

In [4]:
print(chat_response.choices[0].message)

ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='StmnYaOHFR2IqNZjkG6HQ8nqAWUK99cN', function=Function(arguments='{"location":"San Francisco","period":1}', name='get_today_weather'), type='function')], reasoning_content='The user is asking for the weather in San Francisco this morning.\nI have the `get_today_weather` function available.\nThe parameters required are:\n- `location`: "San Francisco"\n- `period`: 1 (for morning)\n\nI will call the function with these arguments.\n')


In [10]:
response_message = chat_response.choices[0].message
if response_message.tool_calls:
    # 1. Append the assistant's tool-call message to the conversation
    messages.append(response_message)

    for tool_call in response_message.tool_calls:
        if tool_call.function.name == "get_today_weather":
            # 2. Actually run your function here
            result = "Sunny, 24°C"  # replace with your real implementation

            print(json.loads(tool_call.function.arguments))
            print('Found tool!')
            # 3. Append the tool result, matching tool_call_id
            # messages.append({
            #     "role": "tool",
            #     "tool_call_id": tool_call.id,
            #     "content": result,
            # })

{'location': 'San Francisco', 'period': 1}
Found tool!


In [5]:
final_response = client.chat.completions.create(
    model="Ternary-Bonsai-27B-Q2_0.gguf",
    messages=messages,
    tools=tools,
    max_tokens=4096,
    temperature=0.7,
    top_p=0.95,
    extra_body={"top_k": 20},
)

print(final_response.choices[0].message.content)

Today's weather is sunny with a temperature of 24°C.


In [9]:
import inspect
def my_func(a, c, b=None, d=4):
    pass

dispatch_dict = {'my_func': my_func}

sig = inspect.signature(dispatch_dict['my_func'])
print(sig)
print(sig.parameters)
optional_count = sum(
    1 for param in sig.parameters.values()
    if param.default is not inspect.Parameter.empty
)
print(optional_count)

(a, c, b=None, d=4)
OrderedDict({'a': <Parameter "a">, 'c': <Parameter "c">, 'b': <Parameter "b=None">, 'd': <Parameter "d=4">})
2


In [3]:
from MarketData.yahoo import formatted_historical_data

data_string = formatted_historical_data('NVDA', '2026-08-07', '2026-08-10', '5m')
print(data_string)

                    Datetime        Open        High         Low       Close  \
0  2026-08-07 09:30:00-04:00  222.000000  222.990005  220.660004  222.960007   
1  2026-08-07 09:35:00-04:00  222.955002  223.130005  221.570007  221.684998   
2  2026-08-07 09:40:00-04:00  221.699997  222.539993  221.274994  222.274994   
3  2026-08-07 09:45:00-04:00  222.270004  222.619995  221.800003  222.179993   
4  2026-08-07 09:50:00-04:00  222.210007  223.169998  222.119995  222.777496   
..                       ...         ...         ...         ...         ...   
73 2026-08-07 15:35:00-04:00  222.529907  223.139999  222.464996  222.985001   
74 2026-08-07 15:40:00-04:00  222.970001  223.100006  222.880005  222.934998   
75 2026-08-07 15:45:00-04:00  222.934998  223.039993  222.660004  222.862198   
76 2026-08-07 15:50:00-04:00  222.860001  223.410004  222.750000  223.395905   
77 2026-08-07 15:55:00-04:00  223.380005  224.210007  223.210007  223.979996   

     Volume  Dividends  Stock Splits  


KeyError: 'Date'

In [7]:
import yfinance as yf
ticker_symbol = "AAPL"

# Create a Ticker object
ticker = yf.Ticker(ticker_symbol)

# Fetch historical market data
historical_data = ticker.history(period="1d", interval='5m')  # data for the last year
print("Historical Data:")
print(historical_data)

Historical Data:
                                 Open        High         Low       Close  \
Datetime                                                                    
2026-08-10 09:30:00-04:00  306.829987  307.310394  304.630005  306.630005   
2026-08-10 09:35:00-04:00  306.674988  306.729004  304.899994  305.904999   
2026-08-10 09:40:00-04:00  305.952515  306.500000  305.625000  305.864990   
2026-08-10 09:45:00-04:00  305.850006  306.899994  305.660004  306.890015   
2026-08-10 09:50:00-04:00  306.855011  307.420013  306.118011  306.600006   
2026-08-10 09:55:00-04:00  306.540009  307.220001  306.480011  307.049988   
2026-08-10 10:00:00-04:00  307.040009  307.420013  306.640289  307.325012   
2026-08-10 10:05:00-04:00  307.339996  307.389893  306.250000  306.410004   
2026-08-10 10:10:00-04:00  306.390808  306.559998  305.950012  306.529999   
2026-08-10 10:15:00-04:00  306.510010  307.079987  306.290009  307.059998   
2026-08-10 10:20:00-04:00  307.040009  307.455902  306.7200